In [1]:
import random
import numpy as np
import os
import torch 

def set_seed(seed=24):
    """Setea semilla para reproducibilidad general"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # si usas multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Semilla fijada en: {seed}")

# Llamar a la función
set_seed(24)

Semilla fijada en: 24


In [2]:
import sys
sys.path.append('../')
 
import pandas as pd 
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from plotly import express as px
from tutoriales.utils import plot_confusion_matrix, get_artifact_filename
from json import loads
from joblib import load, dump
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact
from optuna.visualization import plot_param_importances
from optuna.importance import FanovaImportanceEvaluator
from optuna.importance import MeanDecreaseImpurityImportanceEvaluator  
from optuna.importance import PedAnovaImportanceEvaluator
from optuna.visualization import plot_contour

c:\Users\matia\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)

In [4]:
# Paths
BASE_DIR = '../'
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

***Carga del modelo Tabular:*** Elección Stacking de modelos.

In [5]:
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_1__variables_originales.joblib'))
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_2_m0__variables_completas.joblib'))
lgb_dataset = load(os.path.join(PATH_TO_MODELS, 'stacking_modelos_v1.joblib'))




***Carga del modelo de Texto*** 

In [6]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '5.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3", 
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)


[I 2026-05-19 10:50:16,745] Using an existing study with name '01 DistilBert_5.0' instead of creating a new one.


In [7]:
bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

***Carga del modelo de Imágenes***

In [8]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-05-19 10:50:16,936] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [9]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))
#resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_resnet,'test')))

In [10]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [11]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [12]:
#merged_datasets.head()

In [13]:
# Porcentaje de nulos en cada columna
merged_datasets.isnull().mean().mul(100).round(2).rename('% nulos')


PetID                0.00
lgb_pred_score       0.00
AdoptionSpeed        0.00
bert_pred_score      0.10
resnet_pred_score    2.27
Name: % nulos, dtype: float64

In [14]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score']   = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
merged_datasets['lgb_pred_score']    = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]


In [15]:
# Normalizar scores de cada modelo
all_values = [item for sublist in merged_datasets["resnet_pred_score"] for item in sublist]
min_val = min(all_values)
max_val = max(all_values)

def normalizar_lista(lista):
    return [(x - min_val) / (max_val - min_val) for x in lista]

merged_datasets["resnet_pred_score"] = merged_datasets["resnet_pred_score"].apply(normalizar_lista)

all_values1 = [item for sublist in merged_datasets["bert_pred_score"] for item in sublist]
min_val1 = min(all_values1)
max_val1 = max(all_values1)

def normalizar_lista1(lista):
    return [(x - min_val1) / (max_val1 - min_val1) for x in lista]

merged_datasets["bert_pred_score"] = merged_datasets["bert_pred_score"].apply(normalizar_lista1)

all_values2 = [item for sublist in merged_datasets["lgb_pred_score"] for item in sublist]
min_val2 = min(all_values2)
max_val2 = max(all_values2)

def normalizar_lista2(lista):
    return [(x - min_val2) / (max_val2 - min_val2) for x in lista]

merged_datasets["lgb_pred_score"] = merged_datasets["lgb_pred_score"].apply(normalizar_lista2)

# Asegurar que no queden filas sin AdoptionSpeed o sin predicciones válidas
merged_datasets = merged_datasets.dropna(subset=['AdoptionSpeed', 'lgb_pred_score', 'bert_pred_score', 'resnet_pred_score']).reset_index(drop=True)

# Separar en train/test estratificado por AdoptionSpeed
train_merged_datasets, test_merged_datasets = train_test_split(
    merged_datasets,
    test_size=0.1,
    random_state=24,
    stratify=merged_datasets['AdoptionSpeed']
)

print("Train/test split:")
print(train_merged_datasets['AdoptionSpeed'].value_counts(normalize=True).sort_index())
print(test_merged_datasets['AdoptionSpeed'].value_counts(normalize=True).sort_index())
print(f"Train shape: {train_merged_datasets.shape}, Test shape: {test_merged_datasets.shape}")

Train/test split:
AdoptionSpeed
0    0.027418
1    0.206002
2    0.268989
3    0.217488
4    0.280104
Name: proportion, dtype: float64
AdoptionSpeed
0    0.026667
1    0.206667
2    0.270000
3    0.216667
4    0.280000
Name: proportion, dtype: float64
Train shape: (2699, 5), Test shape: (300, 5)


In [16]:
#merged_datasets.head()

### Merging de modelos

Para mejorar el poder predictivo de los modelos anteriormente vistos se decidió realizar una combinación linear del aporte de predicción de cada uno optimizando el peso asignado a cada modelo, de esta forma se logra una mejor generalización sobre datos no nuevos o no analizados. La función objetivo a maximizar fue la métrica Kappa. 

In [17]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    if total_w == 0: return 0
    
    # Cálculo vectorizado para mayor velocidad usando el conjunto de entrenamiento
    lgb_scores = np.stack(train_merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(train_merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(train_merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(train_merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

Versión 6 train-test 30/70 = 0.4271 0.4326
Versión 5 train-test 60/40 = 0.4474 0.3981
Versión 4 train-test 70/30 = 0.4393 0.4101
Versión 2 train-test 80/20 = 0.4434 0.3936
Versión 3 train-test 90/10 = 0.4380 0.3939

In [18]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db-blend.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_stacking_BERT_ResNet_V6",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-05-19 10:50:17,558] A new study created in RDB with name: Ensemble_stacking_BERT_ResNet_V6
[I 2026-05-19 10:50:17,702] Trial 0 finished with value: 0.34852453340506195 and parameters: {'w_lgb': 0.7655291372379157, 'w_bert': 0.908824619393628, 'w_resnet': 0.5988024365883082}. Best is trial 0 with value: 0.34852453340506195.
[I 2026-05-19 10:50:17,808] Trial 1 finished with value: 0.39958508829959416 and parameters: {'w_lgb': 0.9509291562706198, 'w_bert': 0.28278294478073995, 'w_resnet': 0.031235626390259452}. Best is trial 1 with value: 0.39958508829959416.
[I 2026-05-19 10:50:17,895] Trial 2 finished with value: 0.3438212045401917 and parameters: {'w_lgb': 0.7718980856592387, 'w_bert': 0.7843728527136196, 'w_resnet': 0.03421477224872027}. Best is trial 1 with value: 0.39958508829959416.
[I 2026-05-19 10:50:17,998] Trial 3 finished with value: 0.3077212998628701 and parameters: {'w_lgb': 0.0662042378239851, 'w_bert': 0.349397757709732, 'w_resnet': 0.497723866332064}. Best is tri

In [19]:
# Evaluar en el conjunto de test separado
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

lgb_scores_test = np.stack(test_merged_datasets['lgb_pred_score'].values)
bert_scores_test = np.stack(test_merged_datasets['bert_pred_score'].values)
resnet_scores_test = np.stack(test_merged_datasets['resnet_pred_score'].values)

combined_scores_test = (
    (best_params['w_lgb'] / sum_best_w) * lgb_scores_test +
    (best_params['w_bert'] / sum_best_w) * bert_scores_test +
    (best_params['w_resnet'] / sum_best_w) * resnet_scores_test
)

preds_test = np.argmax(combined_scores_test, axis=1)

test_kappa = cohen_kappa_score(test_merged_datasets['AdoptionSpeed'], preds_test, weights='quadratic')

print(f"Mejor Kappa en entrenamiento: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")
print(f"Kappa en test estratificado: {test_kappa:.4f}")

Mejor Kappa en entrenamiento: 0.4379
Pesos óptimos: {'w_lgb': 0.6479415959652853, 'w_bert': 0.1797053013927432, 'w_resnet': 0.9988806518853812}
Kappa en test estratificado: 0.3873


In [20]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

Mejor Kappa: 0.4379
Pesos óptimos: {'w_lgb': 0.6479415959652853, 'w_bert': 0.1797053013927432, 'w_resnet': 0.9988806518853812}


In [21]:
best_kappa = study_blend.best_trial.value
print(f"Mejor puntuación Kappa: {best_kappa}")

Mejor puntuación Kappa: 0.43788967704614623


In [22]:
# Crear la columna de predicción final optimizada

def _prediction_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.zeros(5, dtype=float)

merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * _prediction_vector(r['lgb_pred_score']) +
    (best_params['w_bert'] / sum_best_w) * _prediction_vector(r['bert_pred_score']) +
    (best_params['w_resnet'] / sum_best_w) * _prediction_vector(r['resnet_pred_score'])
    for _, r in merged_datasets.iterrows()
]

In [23]:
#merged_datasets[['lgb_pred_score', 'bert_pred_score', 'resnet_pred_score']]
merged_datasets['blend_pred_score']

0       [0.2254703499432196, 0.48810088714417776, 0.50...
1       [0.20009991059205384, 0.41677609396927773, 0.3...
2       [0.1856384488991517, 0.3985015538754729, 0.531...
3       [0.0812053499079636, 0.2560486951068473, 0.426...
4       [0.1135496653964571, 0.3826879895572703, 0.597...
                              ...                        
2994    [0.15843305407278502, 0.36698324653402914, 0.3...
2995    [0.14868603836218594, 0.4125022546974358, 0.50...
2996    [0.13069555891329193, 0.5721331241518977, 0.52...
2997    [0.19667388463667634, 0.43801826235880936, 0.4...
2998    [0.12606846427643117, 0.45795631105246215, 0.5...
Name: blend_pred_score, Length: 2999, dtype: object

In [24]:
#merged_datasets['blend_pred_score']

In [25]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred_score'].apply(np.argmax), 
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))

In [26]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred_score'].apply(np.argmax), 
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['bert_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [27]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred_score'].apply(np.argmax), 
                    title = 'ResNet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['resnet_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [28]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blend_pred_score'].apply(np.argmax), 
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['blend_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


### Conclusión

Como puede observarse en cada matriz de confusión donde se indica el Kappa obtenido por cada modelo, para el modelo blended el valor de Kappa es superior a los anteriores, llegando al valor de 0.4343.
